# 강의 05 · 실습 3 — 운영 장치 · (1) 강사 시연


## 1. 문제상황

- 놀이공원 구름월드의 안내 서비스는 모델 하나를 직접 불러 답을 만듭니다. 지난주에 그 모델이 한동안 응답하지 않아 안내가 통째로 멈췄습니다.
- 운영 담당자는 질문 하나에 얼마가 드는지 모르고, 같은 안내문을 매번 통째로 보내면서도 캐시가 되는지 확인해 본 적이 없습니다.
- 하루에 쓸 수 있는 금액 한도를 정해 두어도, 한도를 넘었을 때 호출을 멈추는 장치가 없습니다.
- 쉬운 질문과 어려운 질문에 같은 모델을 쓰고 있어 비용이 필요 이상으로 나갑니다.


## 2. 문제와 목표

- **문제**: 모델 장애에 대비한 대체 경로, 실패의 기록, 호출 비용의 측정, 캐시 확인, 예산 한도, 모델 선택 규칙이 전부 없습니다. 안내 서비스는 답만 내고 운영 정보는 아무것도 남기지 않습니다.
- **목표**
  - 서비스 코드를 고치지 않고 호출 단계에 운영 장치 여섯 개를 붙입니다.
    - 모델 세 개: 기본 `openai/gpt-5.6-luna`, 경량 `openai/gpt-4o-mini`, 고성능 `openai/gpt-5.6-terra`
    - 대체(fallback): 1차 모델이 없을 때 2차 경량 → 3차 고성능으로 넘어감
    - 실패 런 기록: 대체 없이 실패한 호출이 LangSmith에 실패 런으로 남음
    - 비용 측정: 모델 세 개의 입력·출력 토큰과 비용을 표로 출력함
    - 프롬프트 캐싱: 같은 긴 안내문(FAQ 열두 번 반복)을 두 번 보냈을 때의 캐시 히트 확인
    - 토큰 예산: 누적 비용이 0.0003달러 이상이면 호출을 멈춤
    - 티어링: 질문이 30자보다 길거나 「비교」「설명」「추천」이 있으면 고성능, 아니면 경량 모델
- **목표 달성 여부의 판정 기준**:
  - ① 존재하지 않는 1차 모델로 불러도 2차 모델이 답하고 답한 모델 이름이 출력됩니다.
  - ② 대체 없이 부른 실패가 예외 이름과 함께 출력되고 LangSmith에 실패 런으로 남습니다.
  - ③ 모델 세 개의 입력·출력 토큰과 비용이 표로 출력됩니다.
  - ④ 같은 긴 안내문의 2회차 호출에서 캐시 히트 토큰이 0보다 큽니다.
  - ⑤ 예산을 넘긴 회차에서 `BudgetExceededError`가 출력되고 반복이 멈춥니다.
  - ⑥ 질문마다 고른 모델 이름과 비용이 출력되고, 짧은 질문은 경량 모델, 긴 비교 질문은 고성능 모델이 골라집니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex03_s1_diagram.svg)


## 4. 단계별 요구사항

1. **대체(fallback)를 붙입니다.**
    - `completion_with_ladder(messages, primary)`는 `litellm.completion`을 `fallbacks=[CHEAP, HIGH]`와 함께 불러, 1차 모델이 실패하면 2차 경량 모델, 2차도 실패하면 3차 고성능 모델로 넘어갑니다.
    - 시연에서는 1차 모델로 존재하지 않는 이름 `GHOST`를 넣어 대체가 발동하는지 보고, 답한 모델 이름(`res.model`)을 출력합니다.
2. **실패 런을 남깁니다.**
    - 대체 없이 `GHOST`를 부르고 예외를 `try`/`except`로 받아 예외 이름과 메시지 앞부분을 출력합니다.
    - 호출 단계에 붙은 관측 계측이 이 호출을 실패 런으로 LangSmith에 남깁니다.
    - 마지막에 `Client().flush()`를 부릅니다.
3. **비용을 잽니다.**
    - 모델 세 개(`CHEAP`·`PRIMARY`·`HIGH`)를 같은 질문으로 한 번씩 부르고, 모델마다 입력 토큰·출력 토큰·`litellm.completion_cost(res)` 비용을 한 줄씩 표로 출력합니다.
4. **캐시 히트를 확인합니다.**
    - FAQ를 열두 번 반복해 붙인 긴 안내문(`LONG_GUIDE`)을 시스템 메시지로 넣어 같은 질문을 두 번 부르고, 회차마다 입력 토큰·캐시 히트 토큰(`usage`의 `prompt_tokens_details.cached_tokens`)·비용을 출력합니다.
5. **예산 장치를 붙입니다.**
    - `guarded_completion(**kwargs)`는 누적 비용 `spent`가 예산 `BUDGET`(0.0003달러) 이상이면 호출하지 않고 `litellm.BudgetExceededError(current_cost=spent, max_budget=BUDGET)`를 내고, 아니면 호출한 뒤 `completion_cost`를 누적합니다.
    - 같은 질문을 최대 열 번 반복해 몇 회차에서 멈추는지 출력합니다.
6. **티어링을 붙입니다.**
    - `choose_model(question)`은 질문이 30자보다 길거나 「비교」「설명」「추천」이 들어 있으면 `HIGH`, 아니면 `CHEAP`를 돌려줍니다.
    - 질문 세 개를 고른 모델로 부르고 모델 이름·답 앞부분·비용을 출력한 뒤 `Client().flush()`를 부릅니다.


## 5. 코드 골격 — 운영 장치 6단

이미 도는 서비스의 호출 단계에 운영 장치를 붙이는 순서는 다음 여섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 여섯 단계와 하나씩 대응합니다. 여섯 단계 어디에서도 서비스 코드를 고치지 않습니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 대체(fallback) | 1차 모델이 실패하면 2차·3차로 넘어갑니다 | `litellm.completion(..., fallbacks=[CHEAP, HIGH])` | 1 |
| ② 실패 런 기록 | 대체 없는 실패를 예외로 받고 실패 런으로 남깁니다 | `try`/`except`, `Client().flush()` | 2 |
| ③ 비용 측정 | 모델마다 토큰과 비용을 잽니다 | `litellm.completion_cost(res)`, `res.usage` | 3 |
| ④ 프롬프트 캐싱 | 같은 긴 접두를 두 번 보내 캐시 히트를 확인합니다 | `usage.prompt_tokens_details.cached_tokens` | 4 |
| ⑤ 토큰 예산 | 누적 비용이 예산을 넘으면 호출을 멈춥니다 | `litellm.BudgetExceededError` | 5 |
| ⑥ 티어링 | 질문에 따라 경량·고성능 모델을 고릅니다 | `choose_model(question)` | 6 |


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델 이름 세 개를 정합니다. 이 실습의 모델 호출은 `litellm.completion`을 직접 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- 모델 세 개는 기본 모델(`PRIMARY`), 경량 모델(`CHEAP`), 고성능 모델(`HIGH`)입니다. 모델 이름은 공급자 이름을 앞에 붙인 문자열 그대로 쓰고, 별칭이나 중계 서버는 쓰지 않습니다.
- `litellm.suppress_debug_info = True`와 `logging` 설정 한 줄은 오류가 났을 때 litellm이 화면에 출력하는 안내 배너와 오류 로그를 끕니다. 동작에는 영향이 없습니다. 실패 자체는 단계 ②에서 예외로 받아 직접 출력합니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import logging
import os
import time
import warnings

from dotenv import load_dotenv, find_dotenv

import litellm
from langsmith import Client, traceable
from langsmith.run_helpers import get_current_run_tree

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex03"

PRIMARY = "openai/gpt-5.6-luna"
CHEAP = "openai/gpt-4o-mini"
HIGH = "openai/gpt-5.6-terra"
print("모델 세 개:", PRIMARY, CHEAP, HIGH)

운영 장치를 붙일 안내 서비스입니다. 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. `make_messages`가 질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만들고, 모델 호출 함수 `litellm.completion`은 `traceable`로 감싸 llm 런에 기록되어 있습니다(답한 모델 이름을 런 메타데이터에 적습니다). 서비스가 도는 것을 먼저 확인합니다.


In [ ]:
FAQ = """
[환불] Q: 자유이용권 환불 규정 알려 주세요
A: 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.
[운영] Q: 운영 시간이 어떻게 되나요?
A: 평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다.
[야간] Q: 야간개장은 언제 하나요?
A: 금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.
[주차] Q: 주차 요금은 얼마인가요?
A: 자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.
"""

GUIDE = ("너는 놀이공원 구름월드의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
         "인사말에는 짧은 인사로 답한다. FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다.")


def make_messages(question: str, guide: str = GUIDE) -> list:
    """질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만든다."""
    return [{"role": "system", "content": guide + "\n=== FAQ ===\n" + FAQ},
            {"role": "user", "content": question}]


_completion = litellm.completion


@traceable(run_type="llm", name="litellm.completion", metadata={"ls_provider": "openai"})
def completion(**kwargs):
    """이 실습에서 쓰는 관측 계측. 답한 모델 이름을 런 메타데이터에 적는다."""
    res = _completion(**kwargs)
    get_current_run_tree().metadata["ls_model_name"] = res.model
    return res


litellm.completion = completion

Q = "자유이용권 환불이 되나요?"
res = litellm.completion(model=PRIMARY, messages=make_messages(Q))
print(res.model, "→", res.choices[0].message.content[:60])

### 단계 ① — 대체(fallback) (요구사항 1)

`fallbacks`는 1차 모델 호출이 실패했을 때 차례로 시도할 모델 목록(fallbacks)입니다. 존재하지 않는 모델 이름을 1차에 넣으면 실패가 즉시 나므로 대체가 발동하는 것을 바로 볼 수 있습니다. 답한 모델은 응답의 `model` 키에 적혀 있습니다.


In [ ]:
GHOST = "openai/gpt-5.6-luna-nonexistent"   # 1차 모델 장애를 모의로 만드는 존재하지 않는 이름
LADDER = [CHEAP, HIGH]                         # 2차 경량 → 3차 고성능


def completion_with_ladder(messages: list, primary: str = PRIMARY):
    """1차 모델이 실패하면 대체 순서를 따라 2차·3차 모델로 넘어간다."""
    return litellm.completion(model=primary, messages=messages, fallbacks=LADDER)


res = completion_with_ladder(make_messages(Q), primary=GHOST)
print("1차:", GHOST)
print("답한 모델:", res.model)
print(res.choices[0].message.content[:60])

### 단계 ② — 실패 런 기록 (요구사항 2)

대체가 없으면 실패는 예외로 올라옵니다. 예외를 받아 화면에 출력하되, 호출 단계에 붙은 관측 계측이 같은 호출을 LangSmith에 실패 런으로 남깁니다. 실패 런에는 예외 메시지가 그대로 붙어, 나중에 무엇이 왜 실패했는지 화면에서 찾을 수 있습니다.


In [ ]:
try:
    litellm.completion(model=GHOST, messages=make_messages(Q))
    print("예상 밖 성공")
except Exception as e:
    print("실패 런:", type(e).__name__, "-", str(e)[:80])

Client().flush()
print("실패 런이 LangSmith 프로젝트 sesac-lec05-ex03에 남았습니다.")

### 단계 ③ — 비용 측정 (요구사항 3)

`litellm.completion_cost(res)`는 응답의 토큰 수와 모델의 가격표로 비용을 계산합니다. 같은 질문이라도 모델마다 비용이 다르므로, 표로 나란히 놓고 봅니다.


In [ ]:
print(f"{'모델':<22} {'입력 토큰':>6} {'출력 토큰':>6} {'비용(달러)':>12}")
for model in (CHEAP, PRIMARY, HIGH):
    res = litellm.completion(model=model, messages=make_messages(Q))
    cost = litellm.completion_cost(res)
    print(f"{model:<22} {res.usage.prompt_tokens:>8} {res.usage.completion_tokens:>8} {cost:>12.6f}")

### 단계 ④ — 프롬프트 캐싱 (요구사항 4)

같은 긴 접두(시스템 메시지)를 두 번 연속 보내면 2회차에는 접두의 대부분이 캐시에서 읽혀 입력 비용이 줄어듭니다. 캐시가 붙으려면 접두가 충분히 길어야 하므로 FAQ를 열두 번 반복해 넣습니다. 캐시는 몇 분 동안 유지되므로, 접두 맨 앞에 실행 시각 표시를 붙여 실행마다 1회차는 캐시 없이 시작하게 합니다. 캐시 히트 토큰은 응답 `usage`의 `prompt_tokens_details.cached_tokens`에 적혀 있습니다.


In [ ]:
RUN_TAG = time.strftime("%H%M%S")   # 실행마다 접두를 새로 만들어 이전 실행의 캐시가 섞이지 않게 한다
LONG_GUIDE = f"[실행 {RUN_TAG}] " + GUIDE + "\n=== FAQ ===\n" + (FAQ + "\n") * 12

for i in (1, 2):
    res = litellm.completion(model=PRIMARY, messages=[{"role": "system", "content": LONG_GUIDE},
                                                      {"role": "user", "content": Q}])
    usage = res.usage.model_dump()
    cached = (usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0
    print(f"{i}회차: 입력 토큰 {usage['prompt_tokens']} · 캐시 히트 {cached} · 비용 {litellm.completion_cost(res):.6f}")

### 단계 ⑤ — 토큰 예산 (요구사항 5)

예산 장치는 호출 단계를 한 번 더 감싸 누적 비용을 셉니다. 누적이 예산 이상이면 호출하지 않고 `litellm.BudgetExceededError`를 냅니다. litellm에는 `max_budget` 설정이 있지만 이 환경의 litellm 판(1.97.0)은 응답 객체의 비용을 누적하지 않아 발동하지 않으므로, 누적은 직접 하고 예외 클래스만 litellm의 것을 씁니다.


In [ ]:
BUDGET = 0.0003   # 달러
spent = 0.0


def guarded_completion(**kwargs):
    """누적 비용이 예산 이상이면 호출하지 않고 멈춘다. 호출했으면 비용을 누적한다."""
    global spent
    if spent >= BUDGET:
        raise litellm.BudgetExceededError(current_cost=spent, max_budget=BUDGET)
    res = litellm.completion(**kwargs)
    spent += litellm.completion_cost(res)
    return res


for i in range(1, 11):
    try:
        guarded_completion(model=PRIMARY, messages=make_messages(Q))
        print(f"{i}회차 OK · 누적 {spent:.6f} / 예산 {BUDGET}")
    except litellm.BudgetExceededError:
        print(f"{i}회차 멈춤: BudgetExceededError · 누적 {spent:.6f} / 예산 {BUDGET}")
        break

### 단계 ⑥ — 티어링 (요구사항 6)

티어링은 호출 전에 질문을 보고 모델을 고르는 규칙입니다. 짧고 사실을 묻는 질문은 경량 모델로, 길거나 비교·설명을 요구하는 질문은 고성능 모델로 보냅니다. 규칙은 문자열 검사만으로 정하며 모델을 부르지 않습니다.


In [ ]:
def choose_model(question: str) -> str:
    """짧고 사실을 묻는 질문은 경량 모델, 길거나 비교·설명·추천을 요구하는 질문은 고성능 모델."""
    if len(question) > 30 or any(word in question for word in ("비교", "설명", "추천")):
        return HIGH
    return CHEAP


QUESTIONS = [
    "주차 요금은 얼마인가요?",
    "야간개장은 언제 하나요?",
    "평일 오후에 아이 둘과 가는데 자유이용권과 야간개장 중 어느 쪽이 유리한지 비교해서 설명해 주세요.",
]
for question in QUESTIONS:
    model = choose_model(question)
    res = litellm.completion(model=model, messages=make_messages(question))
    print(f"[{model.split('/')[-1]:<13}] {question[:22]} → {res.choices[0].message.content[:36]} ({litellm.completion_cost(res):.6f}달러)")

Client().flush()

## 7. 실행 결과 확인

위 실행 결과에서 다음 여섯 가지를 확인합니다.

1. 단계 ①에서 1차가 존재하지 않는 모델인데도 답한 모델이 `gpt-4o-mini`로 출력됩니다. 2차 모델이 받았다는 뜻입니다.
2. 단계 ②에서 「실패 런: NotFoundError」가 출력됩니다. LangSmith 프로젝트 `sesac-lec05-ex03`에 같은 이름의 런이 빨간 실패 상태로 남습니다.
3. 단계 ③ 표에서 세 모델의 비용이 서로 다르고, 고성능 모델의 비용이 가장 큽니다.
4. 단계 ④에서 1회차의 캐시 히트는 0이고 2회차의 캐시 히트는 0보다 큽니다. 2회차 비용이 1회차보다 작습니다.
5. 단계 ⑤에서 몇 회차 OK가 출력된 뒤 「멈춤: BudgetExceededError」가 출력되고 반복이 끝납니다.
6. 단계 ⑥에서 짧은 질문 두 개는 `gpt-4o-mini`, 긴 비교 질문은 `gpt-5.6-terra`로 출력되고, 고성능 모델의 비용이 가장 큽니다.

서비스 코드는 한 줄도 고치지 않았습니다. 여섯 장치는 전부 호출 단계 바깥에 붙었습니다.
